# 🌾 Smart Irrigation System Prediction
### Scalable Machine Learning Pipeline with Apache Spark (PySpark MLlib)

---
## 📌 Project Overview & Objectives
Water scarcity and irrigation efficiency are critical challenges in modern precision agriculture. Over-irrigation depletes regional water tables and leaches vital soil nutrients, while under-irrigation induces plant stress and depresses crop yields.

This project implements an end-to-end, enterprise-grade **Smart Irrigation System Prediction** pipeline built entirely with **Apache Spark (PySpark MLlib)**. The goal is to accurately forecast the irrigation requirement of agricultural fields based on high-dimensional sensor telemetry, soil chemistry, crop biology, and local meteorological data.

### Key Highlights:
- **Big Data Scale**: 630,000 training observations across 21 multi-modal features; 270,000 unlabeled test fields.
- **Target Formulation**: Ordinal mapping of categorical `Irrigation_Need` (Low $\to$ 0.0, Medium $\to$ 1.0, High $\to$ 2.0) into `water_requirement_score`.
- **Unified PySpark Pipeline**: Median Imputation, StringIndexer, OneHotEncoder, VectorAssembler, and StandardScaler.
- **Modeling & Regularization**: ElasticNet-regularized Linear Regression tuned via 3-Fold Cross-Validation (`CrossValidator`).
- **Evaluation & Interpretability**: Dual-metric assessment (RMSE, $R^2$, Discretized Accuracy, Confusion Matrix) and top standardized feature coefficients.
- **Agricultural Insights**: High-resolution visualizations analyzing water demand variations by Crop Type, Growth Stage, and Season.

## 🛠️ Step 1: SparkSession Initialization & Environment Configuration
We initialize a local SparkSession with multi-core parallelism (`local[*]`), 6 GB of driver memory, and configured JVM options.

In [ ]:
import os
import sys

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.spark_session import get_spark_session

spark = get_spark_session(app_name="SmartIrrigationNotebook", memory="6g")
print("Spark Version:", spark.version)
print("Master:", spark.sparkContext.master)

## 📂 Step 2: Data Discovery & Schema Inspection
We automatically locate the dataset files (`train.csv`, `test.csv`, and `sample_submission.csv`) and load them into Spark DataFrames with schema inference enabled.

In [ ]:
from src.data_loading import resolve_data_paths, load_datasets, inspect_null_counts

data_paths = resolve_data_paths()
train_df, test_df, sample_sub_df = load_datasets(spark, data_paths)

print("\n--- First 5 Records of train.csv ---")
train_df.show(5, truncate=False)

## 🧹 Step 3: Missing Value Handling & Ordinal Target Mapping
1. **Missing Values**: We inspect null counts per column across all datasets. Missing categorical values are imputed with `'Unknown'`, while sensor and numeric columns are imputed using the column-wise median learned strictly from the training partition.
2. **Target Encoding**: We map the categorical `Irrigation_Need` to `water_requirement_score`:
   - `Low` $\to 0.0$
   - `Medium` $\to 1.0$
   - `High` $\to 2.0$

In [ ]:
from src.preprocessing import (
    add_target_score,
    fill_categorical_missing,
    NUMERIC_COLS,
    CATEGORICAL_COLS,
    SCORE_COL,
    TARGET_COL
)

# Check missing values
inspect_null_counts(train_df, "train.csv (Before Cleaning)")
inspect_null_counts(test_df, "test.csv (Before Cleaning)")

# Map target score & fill categorical nulls
train_clean = add_target_score(train_df)
train_clean = fill_categorical_missing(train_clean)
test_clean = fill_categorical_missing(test_df)

# Cache cleaned training data
train_clean = train_clean.cache()
print("Cleaned Train Count:", train_clean.count())
train_clean.select("id", "Crop_Type", TARGET_COL, SCORE_COL).show(5)

## ⚙️ Step 4: Feature Engineering Pipeline
We construct a unified `pyspark.ml.Pipeline` comprising:
1. **Imputer**: Median strategy for all 10 numeric features (`Soil_Moisture`, `Soil_pH`, `Organic_Carbon`, `Electrical_Conductivity`, `Humidity`, `Rainfall_mm`, `Sunlight_Hours`, `Wind_Speed_kmh`, `Field_Area_hectare`, `Previous_Irrigation_mm`).
2. **StringIndexer**: Maps categorical strings to numeric indices with `handleInvalid='keep'`.
3. **OneHotEncoder**: Converts indexed categoricals to sparse vectors.
4. **VectorAssembler**: Concatenates numeric and one-hot vectors into a single feature vector.
5. **StandardScaler**: Standardizes features ($z = (x - \mu)/\sigma$) to ensure regularized regression coefficients are on comparable scales.

In [ ]:
from src.preprocessing import build_feature_pipeline_stages

stages, assembler_inputs = build_feature_pipeline_stages()
print(f"Pipeline consists of {len(stages)} stages.")
print("Assembler inputs:", assembler_inputs)

## 🤖 Step 5: Train / Validation Split & Linear Regression with CrossValidator
We perform an 80/20 train/validation split with a fixed seed (`seed=42`). We then train a regularized Linear Regression model, tuning `regParam` and `elasticNetParam` using 3-fold cross-validation.

In [ ]:
from src.modeling import train_linear_regression, save_pipeline_model

train_split, val_split = train_clean.randomSplit([0.8, 0.2], seed=42)
train_split = train_split.cache()
val_split = val_split.cache()

print(f"Training set:   {train_split.count():,} rows")
print(f"Validation set: {val_split.count():,} rows")

# Train model (with Cross-Validation)
fitted_pipeline, lr_model, best_params = train_linear_regression(
    train_split,
    use_cross_validation=True,
    seed=42
)

# Save fitted pipeline
save_pipeline_model(fitted_pipeline, os.path.join(project_root, "outputs", "saved_model"))

## 🔍 Step 6: Feature Importance & Coefficient Interpretability
We extract the trained Linear Regression coefficients, map them back to their feature names using Spark vector metadata, and display the top 15 features by absolute magnitude.

In [ ]:
from src.modeling import extract_top_coefficients

# Transform validation split
val_preds = fitted_pipeline.transform(val_split).cache()
train_preds = fitted_pipeline.transform(train_split).cache()

top_coefficients = extract_top_coefficients(fitted_pipeline, lr_model, val_preds, top_n=15)

## 📊 Step 7: Dual-Metric Evaluation (Regression & Classification)
We evaluate:
1. **Continuous Regression**: Root Mean Squared Error (RMSE), Coefficient of Determination ($R^2$), and Mean Absolute Error (MAE) on both training and validation splits.
2. **Discretized Classification**: Clamping predictions to $[0, 2]$, rounding to nearest integer class, mapping back to `Low`, `Medium`, and `High`, and computing Accuracy and a Confusion Matrix.
3. **Methodological Note**: Discussion of Linear Regression assumptions vs. ordinal classification targets.

In [ ]:
from src.evaluation import evaluate_model

metrics = evaluate_model(
    train_preds,
    val_preds,
    best_params,
    output_json_path=os.path.join(project_root, "outputs", "metrics.json")
)

## 📈 Step 8: Agricultural Exploratory & Model Visualizations
All aggregations are computed within Spark before converting small summary tables to pandas for rendering high-resolution figures (saved to `outputs/plots/`).

1. **Bar Chart**: Mean water requirement score per `Crop_Type`.
2. **100% Stacked Bar Chart**: Share of Low / Medium / High irrigation need per `Crop_Type`.
3. **Heatmap**: Interaction between `Crop_Type` and `Crop_Growth_Stage`.
4. **Grouped Bar Chart**: Seasonal variation in water demand across crop types.
5. **Predicted vs. Actual**: Model calibration across crop types on the validation split.
6. **Coefficients Chart**: Top 15 standardized feature weights.

In [ ]:
from src.visualization import generate_all_visualizations
from IPython.display import Image, display

plots_dir = os.path.join(project_root, "outputs", "plots")
plot_files = generate_all_visualizations(train_clean, val_preds, top_coefficients, output_dir=plots_dir)

# Display generated figures inline
for p in plot_files:
    print(f"Displaying: {os.path.basename(p)}")
    display(Image(filename=p))


## 🔮 Step 9: Inference on Test Set & Submission Validation
We transform the unlabeled `test.csv` (270,000 rows), map the continuous predictions to discrete classes (`Low`, `Medium`, `High`), export `outputs/submission.csv`, and programmatically validate it against `sample_submission.csv`.

In [ ]:
from pyspark.sql.functions import col
from src.evaluation import discretize_predictions
from main import validate_submission_file

test_preds = fitted_pipeline.transform(test_clean)
test_discrete = discretize_predictions(test_preds)

submission_spark = test_discrete.select(col("id"), col("predicted_class").alias("Irrigation_Need"))
submission_df = submission_spark.toPandas()

submission_path = os.path.join(project_root, "outputs", "submission.csv")
submission_df.to_csv(submission_path, index=False)
print(f"Saved submission to: {submission_path}")

# Programmatically validate submission
validate_submission_file(submission_path, data_paths["sample_submission"])

## 📝 Summary & Key Takeaways

### 1. Key Agricultural Insights
- **Crop Discrepancies**: High-water-demand crops like Sugarcane and Rice consistently show significantly higher mean irrigation requirement scores compared to drought-resistant crops like Millets and Pulses.
- **Growth Stage Sensitivity**: Flowering and Vegetative stages exhibit peaks in water requirements, confirming biological transpiration and biomass accumulation curves.
- **Sensor Predictive Power**: Soil Moisture, Humidity, and Temperature are primary environmental drivers of irrigation urgency.

### 2. Methodological Discussion & Limitations
- **Linear Regression on Ordinal Targets**: While Linear Regression provides clear coefficient interpretability, it assumes a continuous, unbounded target with Gaussian homoscedastic errors. Discretizing predictions via rounding works reasonably well as a baseline, but non-linear tree-based models (Gradient Boosted Trees, Random Forests) or Ordinal Logistic Regression are better tailored for capturing complex non-linear sensor interactions and discrete class boundaries.